# GGS 553 Final GIS Workflow
## Spatial Analysis of Traffic Crash Hotspots in Fairfax County, Virginia

This notebook reproduces the final GIS workflow for the project: data preparation, KDE, hotspot overlay, crash summaries, and OLS preparation.

In [ ]:
import arcpy
from arcpy.sa import KernelDensity, Slice, Reclassify, RemapRange, ZonalStatisticsAsTable
import os

arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

workspace = r"C:\Users\dveronez\OneDrive - George Mason University - O365 Production\PhD_master\2026 - Geographic Information Systems (GGS-553-001, GGS-553-P01)\week4\week4\lab2\project.gdb"

crash_fc = r"C:\Users\dveronez\OneDrive - George Mason University - O365 Production\PhD_master\2026 - Geographic Information Systems (GGS-553-001, GGS-553-P01)\week4\week4\lab2\lab2.gdb\CrashData_Basic_Project"
boundary_fc = r"C:\Users\dveronez\OneDrive - George Mason University - O365 Production\PhD_master\2026 - Geographic Information Systems (GGS-553-001, GGS-553-P01)\week4\week4\lab2\lab2.gdb\FairfaxCountyBoundary_"
tract_fc = r"C:\Users\dveronez\OneDrive - George Mason University - O365 Production\PhD_master\2026 - Geographic Information Systems (GGS-553-001, GGS-553-P01)\week4\week4\lab2\lab2.gdb\census_Fairfax_C_SpatialJoin"

projected_sr = arcpy.SpatialReference(2283)
arcpy.env.workspace = workspace

summary_folder = r"C:\Users\dveronez\OneDrive - George Mason University - O365 Production\PhD_master\2026 - Geographic Information Systems (GGS-553-001, GGS-553-P01)\week4\week4\lab2\outputs"
if not os.path.exists(summary_folder):
    os.makedirs(summary_folder)

print("Workspace:", workspace)
print("Output folder:", summary_folder)


## 1. Output names

In [ ]:
boundary_prj = os.path.join(workspace, "fairfax_boundary_prj")
crash_prj = os.path.join(workspace, "crash_prj")
tract_prj = os.path.join(workspace, "tract_prj")

crash_clip = os.path.join(workspace, "crash_clip")
kde_raster = os.path.join(workspace, "fairfax_kde")
tract_out = os.path.join(workspace, "tracts_with_kde")
kde_table = os.path.join(workspace, "tract_kde_table")

kde_slice = os.path.join(workspace, "fairfax_kde_slice10")
hotspot_raster = os.path.join(workspace, "fairfax_hotspot_raster")
hotspot_poly = os.path.join(workspace, "fairfax_hotspot_poly")
hotspot_poly_single = os.path.join(workspace, "fairfax_hotspot_poly_single")
crash_hotspots = os.path.join(workspace, "crashes_in_hotspots")


## 2. Project data and clip crashes to Fairfax County

In [ ]:
for src, out_fc in [(boundary_fc, boundary_prj), (crash_fc, crash_prj), (tract_fc, tract_prj)]:
    if arcpy.Exists(out_fc):
        arcpy.management.Delete(out_fc)
    arcpy.management.Project(src, out_fc, projected_sr)

if arcpy.Exists(crash_clip):
    arcpy.management.Delete(crash_clip)

arcpy.analysis.Clip(crash_prj, boundary_prj, crash_clip)

print("Crash count inside Fairfax:", arcpy.management.GetCount(crash_clip)[0])
print("Created:", crash_clip)


## 3. Kernel Density Estimation (KDE)

Parameters: cell size = 250 feet; search radius = 1000 feet; density units = square kilometers.

In [ ]:
cell_size = 250
search_radius = 1000

if arcpy.Exists(kde_raster):
    arcpy.management.Delete(kde_raster)

kde = KernelDensity(
    in_features=crash_clip,
    population_field=None,
    cell_size=cell_size,
    search_radius=search_radius,
    area_unit_scale_factor="SQUARE_KILOMETERS",
    out_cell_values="DENSITIES",
    method="PLANAR"
)

kde.save(kde_raster)
print("KDE raster created:", kde_raster)


## 4. Create tract dataset with crash density and mean KDE

In [ ]:
if arcpy.Exists(tract_out):
    arcpy.management.Delete(tract_out)

arcpy.management.CopyFeatures(tract_prj, tract_out)

if "crash_density_km2" not in [f.name for f in arcpy.ListFields(tract_out)]:
    arcpy.management.AddField(tract_out, "crash_density_km2", "DOUBLE")

expr = """
def calc_density(join_count, aland):
    if aland is None or aland == 0:
        return None
    return float(join_count) / (float(aland) / 1000000.0)
"""

arcpy.management.CalculateField(
    tract_out,
    "crash_density_km2",
    "calc_density(!Join_Count!, !ALAND!)",
    "PYTHON3",
    expr
)

if arcpy.Exists(kde_table):
    arcpy.management.Delete(kde_table)

ZonalStatisticsAsTable(
    in_zone_data=tract_out,
    zone_field="GEOID",
    in_value_raster=kde_raster,
    out_table=kde_table,
    ignore_nodata="DATA",
    statistics_type="MEAN"
)

arcpy.management.JoinField(tract_out, "GEOID", kde_table, "GEOID", ["MEAN"])

if "mean_kde" not in [f.name for f in arcpy.ListFields(tract_out)]:
    arcpy.management.AddField(tract_out, "mean_kde", "DOUBLE")

arcpy.management.CalculateField(tract_out, "mean_kde", "!MEAN!", "PYTHON3")

print("Tract dataset ready:", tract_out)


## 5. Extract KDE-based hotspot areas

The KDE raster is divided into 10 equal-area classes. The highest class is used as the hotspot zone.

In [ ]:
if arcpy.Exists(kde_slice):
    arcpy.management.Delete(kde_slice)

out_slice = Slice(kde_raster, 10, "EQUAL_AREA")
out_slice.save(kde_slice)

if arcpy.Exists(hotspot_raster):
    arcpy.management.Delete(hotspot_raster)

reclass = Reclassify(
    kde_slice,
    "Value",
    RemapRange([[1, 9, "NODATA"], [10, 10, 1]]),
    "NODATA"
)

reclass.save(hotspot_raster)

print("KDE slice created:", kde_slice)
print("Hotspot raster created:", hotspot_raster)


## 6. Convert hotspot raster to polygons

In [ ]:
if arcpy.Exists(hotspot_poly):
    arcpy.management.Delete(hotspot_poly)

arcpy.conversion.RasterToPolygon(
    in_raster=hotspot_raster,
    out_polygon_features=hotspot_poly,
    simplify="SIMPLIFY",
    raster_field="Value"
)

if arcpy.Exists(hotspot_poly_single):
    arcpy.management.Delete(hotspot_poly_single)

arcpy.management.MultipartToSinglepart(
    in_features=hotspot_poly,
    out_feature_class=hotspot_poly_single
)

if "area_km2" not in [f.name for f in arcpy.ListFields(hotspot_poly_single)]:
    arcpy.management.AddField(hotspot_poly_single, "area_km2", "DOUBLE")

arcpy.management.CalculateGeometryAttributes(
    hotspot_poly_single,
    [["area_km2", "AREA_GEODESIC"]],
    area_unit="SQUARE_KILOMETERS"
)

print("Hotspot polygons:", arcpy.management.GetCount(hotspot_poly_single)[0])
print("Created:", hotspot_poly_single)


## 7. Create crash points inside hotspots

This is the correct dataset for overlay summaries because each row is one crash inside the KDE hotspot zone.

In [ ]:
if arcpy.Exists(crash_hotspots):
    arcpy.management.Delete(crash_hotspots)

arcpy.analysis.SpatialJoin(
    target_features=crash_clip,
    join_features=hotspot_poly_single,
    out_feature_class=crash_hotspots,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_COMMON",
    match_option="INTERSECT"
)

print("Crashes inside hotspots:", arcpy.management.GetCount(crash_hotspots)[0])
print("Created:", crash_hotspots)


## 8. Overlay summary tables

In [ ]:
def create_frequency(input_fc, field_name, out_name):
    out_table = os.path.join(workspace, out_name)

    if arcpy.Exists(out_table):
        arcpy.management.Delete(out_table)

    arcpy.analysis.Frequency(input_fc, out_table, [field_name])
    print(f"Created: {out_table}")
    return out_table


summary_fields = {
    "Weather Condition": "WEATHER_CONDITION",
    "Light Condition": "LIGHT_CONDITION",
    "Roadway Alignment": "ROADWAY_ALIGNMENT",
    "Traffic Control Type": "TRAFFIC_CONTROL_TYPE",
    "Alcohol Involvement": "ALCOHOL_NOTALCOHOL",
    "Persons Injured": "PERSONS_INJURED",
    "Work Zone Location": "WORK_ZONE_LOCATION",
    "Crash Severity": "CRASH_SEVERITY",
    "Collision Type": "COLLISION_TYPE",
    "Area Type": "AREA_TYPE",
    "Night": "NIGHT",
    "Senior": "SENIOR_NOTSENIOR",
    "Young": "YOUNG_NOTYOUNG"
}

frequency_tables = {}

for label, field in summary_fields.items():
    clean_label = label.lower().replace(" ", "_")
    out_name = f"freq_hotspots_{clean_label}"
    frequency_tables[label] = create_frequency(crash_hotspots, field, out_name)

print("Frequency tables complete.")

## 9. Summary statistics for numeric fields

In [ ]:
numeric_fields = ["PERSONS_IN", "SPEED_DIFF"]

available_fields = [f.name for f in arcpy.ListFields(crash_hotspots)]
stats_fields = []

for field in numeric_fields:
    if field in available_fields:
        stats_fields.append([field, "SUM"])
        stats_fields.append([field, "MEAN"])
    else:
        print(f"Numeric field not found: {field}")

summary_stats_table = os.path.join(workspace, "summary_hotspots_numeric")

if stats_fields:
    if arcpy.Exists(summary_stats_table):
        arcpy.management.Delete(summary_stats_table)

    arcpy.analysis.Statistics(
        in_table=crash_hotspots,
        out_table=summary_stats_table,
        statistics_fields=stats_fields
    )

    print("Numeric summary table created:", summary_stats_table)
else:
    print("No numeric fields available for summary.")


## 10. Export summary tables to CSV

In [ ]:
def export_table_to_csv(table_path, csv_name):
    if table_path is None or not arcpy.Exists(table_path):
        return

    out_csv = os.path.join(summary_folder, csv_name)

    if os.path.exists(out_csv):
        os.remove(out_csv)

    arcpy.conversion.ExportTable(table_path, out_csv)
    print("Exported:", out_csv)

for label, table in frequency_tables.items():
    csv_name = "hotspot_" + label.lower().replace(" ", "_").replace("-", "_") + ".csv"
    export_table_to_csv(table, csv_name)

if arcpy.Exists(summary_stats_table):
    export_table_to_csv(summary_stats_table, "hotspot_numeric_summary.csv")

print("CSV export complete.")


## 11. OLS preparation placeholder

The next stage will use `tracts_with_kde` as the tract-level dataset. Recommended dependent variable: `crash_density_km2`. Explanatory variables should be aggregated to one row per tract.

In [ ]:
print("Ready for OLS preparation.")
print("Tract-level dataset:", tract_out)
print("Recommended dependent variable: crash_density_km2")
